# MGS-28 : Bare Bones PSO MGS contre mealpy — le PSO sans vitesse, et ses deux ancrages

**Navigation** : [<< MGS-27 (FBI vs mealpy)](MGS-27-ForensicBasedInvestigation-vs-Mealpy.ipynb) | [Index](README.md)

**Kernel** : .NET (C#) — pont PythonNet vers mealpy dans la même exécution

***

## Introduction

MGS-27 a refermé la question FBI sur la paire qui prétendait le moins à la différence : le port C# dérivé battait sa référence mealpy des deux mains. La paire 7 descend d'un cran encore dans la prétention — et change de nature : le **Bare Bones PSO** de Kennedy (2003) est le PSO dépouillé de toute sa mécanique, ni inertie *w*, ni coefficients *c1*/*c2* : chaque gène de la nouvelle particule est un simple **tirage gaussien** autour du milieu entre le meilleur personnel et le meilleur global. Deux conséquences structurelles gouvernent ce protocole :

1. **Aucun PSO de mealpy 3.0.2 n'est bare-bones** — les sept variantes du module (`OriginalPSO`, `P_PSO`, `C_PSO`, `CL_PSO`, `AIW_PSO`, `HPSO_TVAC`, `LDW_PSO`) portent toutes les constantes de vitesse du PSO classique ; en prendre une pour jumeau serait mesurer un **PSO classique déguisé** en bare-bones. Le jumeau mealpy est donc un *subclass* de `Optimizer` — le point d'extension natif de la bibliothèque — qui **pinne la formule de Kennedy** dans le harnais mealpy (graine, comptage d'évaluations, correction de bornes).
2. **Le portage MGS est ancré sur la position courante, pas sur le pbest** — le framework géométrique n'expose pas de mémoire *pbest_i* par particule, et l'écart est documenté *in-source* dans `BareBonesParticleSwarm.cs`. Pour distinguer l'écart **noyau** (moteur contre moteur) de l'écart **d'ancrage** (sémantique du portage), le bench croise donc **trois bras** : le composé MGS, la variante mealpy alignée sur sa sémantique, et le Kennedy 2003 littéral.

La grille, la représentation R1, le budget (population 50, 160 générations) et les graines {0, 1, 7, 42} sont ceux des paires précédentes : les médianes se lisent dans la même colonne du tableau croisé de l'Epic.


***

In [1]:
// === MGS-28 : socle commun — DLLs MGS, grille de référence, fonction de coût ===
// Même socle que MGS-22/23 : la représentation R1 (continu + arrondi) est le substrat du bench.
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
using MetaGeneticSharp;
using GeneticSharp;
using System.Diagnostics;

// Grille facile Easy[0] de Sudoku_Easy51.txt — la MÊME que MGS-21/22/23/25/27.
public static string PuzzleLine28 = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";

public static int[,] ParsePuzzle28()
{
    var g = new int[9, 9];
    for (int i = 0; i < 81; i++) g[i / 9, i % 9] = PuzzleLine28[i] - '0';
    return g;
}

// Fonction de coût du bench : conflits totaux (lignes + colonnes + blocs) sur grille PLEINE.
// Renvoie 0 ssi résolu. Le côté Python réimplémente exactement ce comptage.
public static int CountConflicts28(int[,] g)
{
    int conflicts = 0;
    for (int i = 0; i < 9; i++)
    {
        var row = new HashSet<int>(); var col = new HashSet<int>(); var blk = new HashSet<int>();
        for (int j = 0; j < 9; j++)
        {
            if (!row.Add(g[i, j])) conflicts++;
            if (!col.Add(g[j, i])) conflicts++;
            int br = 3 * (i / 3) + j / 3, bc = 3 * (i % 3) + j % 3;
            if (!blk.Add(g[br, bc])) conflicts++;
        }
    }
    return conflicts;
}

public static int CountEmpty28(int[,] p) { int n = 0; foreach (var v in p) if (v == 0) n++; return n; }

public static List<(int r, int c)> EmptyCells28(int[,] p)
{
    var l = new List<(int, int)>();
    for (int r = 0; r < 9; r++) for (int c = 0; c < 9; c++) if (p[r, c] == 0) l.Add((r, c));
    return l;
}

// Décodage R1 : arrondi + clamp vers 1..9 sur les cellules vides, ordre de lecture.
public static int[,] DecodeR1_28(double[] genes)
{
    var Puzzle = ParsePuzzle28();
    var empties = EmptyCells28(Puzzle);
    var g = (int[,])Puzzle.Clone();
    for (int k = 0; k < empties.Count; k++)
        g[empties[k].r, empties[k].c] = Math.Max(1, Math.Min(9, (int)Math.Round(genes[k])));
    return g;
}

var Puzzle28 = ParsePuzzle28();
Console.WriteLine($"Grille de référence : {CountEmpty28(Puzzle28)} cellules vides, " +
                  $"{81 - CountEmpty28(Puzzle28)} indices fixes, {EmptyCells28(Puzzle28).Count} gènes R1.");


Grille de référence : 36 cellules vides, 45 indices fixes, 36 gènes R1.


**Lecture.** Le socle est posé, identique à MGS-23 au nom près — c'est voulu : la comparabilité
des paires de l'Epic repose sur un protocole strictement apparié (même grille Easy[0], même
représentation R1 continue à 51 gènes, même fonction de coût comptée en conflits, renvoyant 0
ssi la grille est résolue). Le pont PythonNet qui suivra réimplémente ce comptage *à
l'identique* et le vérifie sur des vecteurs témoins déterministes : la seule différence
mesurable entre les bras sera donc le moteur, pas l'échelle de la fitness.


In [2]:
// === Moteur MGS : chromosome R1, fitness instrumentée, composé BareBonesParticleSwarm ===
// Kennedy (2003) "Bare Bones" : chaque gène est échantillonné N((ancre + gbest)/2, |gbest - ancre|).
// Ni inertie w ni c1/c2 — les constantes de Clerc (w=0,7298 ; c1=c2=1,49618) NE S'APPLIQUENT PAS
// à ce composé : un BBPSO qui les porterait serait un PSO classique déguisé.
public class SudokuR1Chromosome28 : ChromosomeBase
{
    private const double LO = 1.0, HI = 10.0;
    public SudokuR1Chromosome28() : base(EmptyCells28(ParsePuzzle28()).Count) { CreateGenes(); }
    public override Gene GenerateGene(int index)
        => new Gene(RandomizationProvider.Current.GetDouble(LO, HI));
    public override IChromosome CreateNew() => new SudokuR1Chromosome28();
    public double[] ToGenes() { var v = new double[Length]; for (int i = 0; i < Length; i++) v[i] = (double)GetGene(i).Value; return v; }
    public int[,] ToGrid() => DecodeR1_28(ToGenes());
}

// Fitness instrumentée : chaque évaluation est comptée — le budget se mesure, il ne se suppose pas.
public class SudokuR1Fitness28 : IFitness
{
    public static int Evals;
    public double Evaluate(IChromosome chromosome)
    {
        Evals++;
        return -CountConflicts28(((SudokuR1Chromosome28)chromosome).ToGrid());
    }
}

public static class Mgs28Host
{
    // Trajectoire best-so-far : conflits du meilleur APRÈS chaque génération (event GenerationRan).
    public static List<int> RunTrajectory(int seed, int popSize, int maxGens,
        out int conflicts, out int evals, out double ms, out double[] genes)
    {
        // Seeding AVANT création de population : le RNG est consommé par CreateNew()
        // de chaque individu initial (leçon #12071 / MGS-21).
        FastRandomRandomization.ResetSeed(seed);
        var compound = MetaHeuristicsService.CreateMetaHeuristicByName(
            "BareBonesParticleSwarm", maxGens, popSize);
        var adam = new SudokuR1Chromosome28();
        var pop = new MetaPopulation(popSize, popSize, adam);
        var ga = new MetaGeneticAlgorithm(
            pop, new SudokuR1Fitness28(),
            new EliteSelection(), new UniformCrossover(0.5f), new UniformMutation(true),
            compound);
        ga.Termination = new GenerationNumberTermination(maxGens);
        var traj = new List<int>(maxGens + 1);
        ga.GenerationRan += (s, e) =>
            traj.Add(CountConflicts28(((SudokuR1Chromosome28)ga.BestChromosome).ToGrid()));
        SudokuR1Fitness28.Evals = 0;
        var sw = Stopwatch.StartNew();
        ga.Start();
        sw.Stop();
        var best = (SudokuR1Chromosome28)ga.BestChromosome;
        conflicts = CountConflicts28(best.ToGrid());
        evals = SudokuR1Fitness28.Evals;
        ms = sw.Elapsed.TotalMilliseconds;
        genes = best.ToGenes();
        return traj;
    }

    // Checkpoints 25/50/75/100 % du budget : trajectoire indexée après la génération correspondante.
    public static (int conflicts, int evals, double ms, double[] genes, int[] cp) RunBbpso(int seed, int popSize, int maxGens)
    {
        var traj = RunTrajectory(seed, popSize, maxGens, out var c, out var e, out var t, out var g);
        var q = new[] { traj[maxGens / 4 - 1], traj[maxGens / 2 - 1], traj[3 * maxGens / 4 - 1], traj[maxGens - 1] };
        return (c, e, t, g, q);
    }
}

// Échauffement JIT (course jetée), puis course témoin graine 7.
var warmupMgs = Mgs28Host.RunBbpso(123, 50, 10);
var demoMgs = Mgs28Host.RunBbpso(7, 50, 160);
Console.WriteLine($"MGS BBPSO (graine 7, témoin) : {demoMgs.Item1} conflits, {demoMgs.Item2} évaluations, " +
                  $"{demoMgs.Item3:F0} ms, checkpoints 25/50/75/100 % = {string.Join("/", demoMgs.Item5)}.");


MGS BBPSO (graine 7, témoin) : 29 conflits, 8000 évaluations, 327 ms, checkpoints 25/50/75/100 % = 29/29/29/29.


**Lecture.** Le composé `BareBonesParticleSwarm` est la transcription directe du **Bare Bones
Particle Swarms** de Kennedy (SIS 2003) dans la grammaire géométrique MGS : l'opérateur
*crossver géométrique* à deux parents lit `[position courante, gbest]` et tire chaque gène
dans `N((x_i + gbest)/2, |gbest - x_i|)` — la formule est **pinnée in-source**
(`DefaultSampleOperator`, Box-Muller sur l'`IRandomization`). Trois propriétés à lire dans le
portage : (1) **pas de mémoire pbest** — l'ancre personnelle est la position *courante*
(variante « gbest-ancrée », écart documenté dans le fichier source, la cellule Formules le
croise) ; (2) **gel au meilleur global** — quand une particule *est* le gbest, l'écart-type
s'annule et son tirage retombe sur elle-même : l'élite ne bouge pas, par construction ; (3)
**sélection élitiste héritée** (`FitnessBasedElitistReinsertion`, best-N parents+enfants) —
le garde-fou qui empêche un mauvais tirage gaussien de détruire l'incumbent. La trajectoire
best-so-far est enregistrée à chaque génération via l'événement `GenerationRan` : les
checkpoints 25/50/75/100 % du budget en sont extraits pour la classification de la forme
d'écart.


In [3]:
// === Le pont PythonNet : mealpy dans le même kernel, la même exécution ===
// Recette validée MGS-22 (#12356) : pythonnet 3.1.0, DLL résolue par probe
// (PYTHONNET_PYDLL d'abord, sinon installs standards par OS — aucun chemin machine en dur).
#r "nuget: pythonnet,3.1.0"
using Python.Runtime;
static string ResolvePythonDll28()
{
    var env = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    if (!string.IsNullOrEmpty(env) && System.IO.File.Exists(env)) return env;
    if (OperatingSystem.IsWindows())
    {
        // Installs CPython.org standards d'abord (les plus récentes portent les packages récents
        // comme mealpy), ensuite le scan LOCALAPPDATA — un Python périmé qui n'a pas mealpy
        // ne doit pas masquer une install plus récente (pb 2026-08-25 : Python310 2023 devant Python313).
        foreach (var c in new[] { @"C:\Python313\python313.dll", @"C:\Python312\python312.dll" })
            if (System.IO.File.Exists(c)) return c;
        var local = Environment.GetEnvironmentVariable("LOCALAPPDATA");
        if (!string.IsNullOrEmpty(local))
        {
            var pyDir = System.IO.Path.Combine(local, "Programs", "Python");
            if (System.IO.Directory.Exists(pyDir))
                foreach (var d in System.IO.Directory.GetDirectories(pyDir, "Python3*"))
                {
                    var hit = System.IO.Directory.GetFiles(d, "python3*.dll");
                    if (hit.Length > 0) return hit[0];
                }
        }
        var home = Environment.GetFolderPath(Environment.SpecialFolder.UserProfile);
        foreach (var root in new[] {
                     System.IO.Path.Combine(home, "miniconda3"),
                     System.IO.Path.Combine(home, "anaconda3"),
                     @"C:\ProgramData\miniconda3",
                     @"C:\ProgramData\anaconda3" })
        {
            if (!System.IO.Directory.Exists(root)) continue;
            var hits = System.IO.Directory.GetFiles(root, "python3*.dll");
            var pick = "";
            foreach (var h in hits)
                if (System.IO.Path.GetFileName(h).Length > 11) pick = h;
            if (pick == "" && hits.Length > 0) pick = hits[0];
            if (pick != "")
            {
                var dirs = new[] { root,
                    System.IO.Path.Combine(root, "Library", "mingw-w64", "bin"),
                    System.IO.Path.Combine(root, "Library", "bin"),
                    System.IO.Path.Combine(root, "Scripts") };
                var path = Environment.GetEnvironmentVariable("PATH") ?? "";
                var toAdd = "";
                foreach (var d in dirs)
                    if (System.IO.Directory.Exists(d) && !path.Contains(d + ";"))
                        toAdd += d + ";";
                if (toAdd != "")
                    Environment.SetEnvironmentVariable("PATH", toAdd + path);
                return pick;
            }
        }
    }
    else
    {
        var libs = new[] { "/usr/lib/x86_64-linux-gnu", "/usr/lib", "/usr/local/lib", "/opt/homebrew/lib" };
        foreach (var dir in libs)
            if (System.IO.Directory.Exists(dir))
            {
                var hit = System.IO.Directory.GetFiles(dir, "libpython3.*");
                foreach (var h in hit)
                    if (h.EndsWith(".so") || h.EndsWith(".dylib")) return h;
            }
    }
    throw new System.IO.FileNotFoundException(
        "DLL Python introuvable : definir PYTHONNET_PYDLL ou installer Python 3.10+ (mealpy requis).");
}
Runtime.PythonDLL = ResolvePythonDll28();
PythonEngine.Initialize();

// Le problème Python : decode + coût réimplémentés à l'identique, compteur d'évals,
// et le JUMEAU BARE-BONES : subclass Optimizer (point d'extension natif mealpy) pinnant
// la formule de Kennedy 2003 — aucune des 7 variantes PSO de mealpy 3.0.2 n'est bare-bones.
public static PyModule S28;
using (Py.GIL())
{
    S28 = Py.CreateScope();
    S28.Set("puzzle_line28", PuzzleLine28);
    S28.Exec(@"import sys
import numpy as np
import mealpy
from mealpy import Problem, FloatVar, Optimizer
from mealpy.swarm_based.PSO import OriginalPSO, P_PSO, C_PSO, CL_PSO, AIW_PSO, HPSO_TVAC, LDW_PSO
import json as _json

puzzle = [int(ch) for ch in puzzle_line28]
empties = [(i // 9, i % 9) for i in range(81) if puzzle[i] == 0]

def decode(vec):
    g = [puzzle[r * 9:(r + 1) * 9] for r in range(9)]
    for k in range(len(empties)):
        r, c = empties[k]
        v = int(round(float(vec[k])))
        g[r][c] = max(1, min(9, v))
    return g

def cost(g):
    conflicts = 0
    for i in range(9):
        units = ([g[i][j] for j in range(9)],
                 [g[j][i] for j in range(9)],
                 [g[3 * (i // 3) + j // 3][3 * (i % 3) + j % 3] for j in range(9)])
        for unit in units:
            seen = set()
            for v in unit:
                if v in seen:
                    conflicts += 1
                seen.add(v)
    return conflicts

def cost_of_vector(vec):
    return cost(decode(vec))

PY_EVALS = [0]

class SudokuProblem(Problem):
    def __init__(self, bounds=None, minmax='min', **kwargs):
        super().__init__(bounds, minmax, log_to='nothing', **kwargs)
    def obj_func(self, x):
        PY_EVALS[0] += 1
        return float(cost(decode(x)))

class OriginalBBPSO(Optimizer):
    # Bare Bones PSO (Kennedy 2003) — jumeau mealpy du composé MGS BareBonesParticleSwarm.
    # Formule pinnée : x_new ~ N((ancre + gbest)/2, |gbest - ancre|) — aucune constante de Clerc.
    #   variant='kennedy' : ancre = pbest_i (local_solution), remplacement INCONDITIONNEL — papier littéral.
    #   variant='mgs'     : ancre = position courante x_i, sélection gloutonne per-particle —
    #                       sémantique alignée sur le composé MGS (ancre Current + réinsertion élitiste).
    def __init__(self, epoch=10000, pop_size=100, variant='kennedy', **kwargs):
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int('epoch', epoch, [1, 100000])
        self.pop_size = self.validator.check_int('pop_size', pop_size, [2, 10000])
        self.variant = variant
        self.checkpoints = {}

    def generate_agent(self, solution=None):
        agent = self.generate_empty_agent(solution)
        agent.target = self.get_target(agent.solution)
        agent.local_solution = agent.solution.copy()
        agent.local_target = agent.target.copy()
        return agent

    def evolve(self, epoch):
        # Checkpoint APRÈS l'epoch m : enregistré à l'entrée de l'epoch m+1 (g_best à jour).
        if epoch - 1 in (self.epoch // 4, self.epoch // 2, 3 * self.epoch // 4):
            self.checkpoints[epoch - 1] = float(self.g_best.target.fitness)
        for idx in range(self.pop_size):
            anchor = self.pop[idx].local_solution if self.variant == 'kennedy' else self.pop[idx].solution
            g = self.g_best.solution
            mean = 0.5 * (anchor + g)
            std = np.abs(g - anchor)
            pos_new = self.correct_solution(self.generator.normal(mean, std))
            target = self.get_target(pos_new)
            if self.variant == 'kennedy':
                self.pop[idx].update(solution=pos_new.copy(), target=target.copy())
            elif self.compare_target(target, self.pop[idx].target, self.problem.minmax):
                self.pop[idx].update(solution=pos_new.copy(), target=target.copy())
            if self.compare_target(target, self.pop[idx].local_target, self.problem.minmax):
                self.pop[idx].update(local_solution=pos_new.copy(), local_target=target.copy())

def run_mealpy_bbpso(seed, pop_size, epoch, variant='kennedy'):
    import time
    PY_EVALS[0] = 0
    prob = SudokuProblem(bounds=FloatVar(lb=(1.0,) * len(empties), ub=(10.0,) * len(empties), name='genes'), minmax='min')
    model = OriginalBBPSO(epoch=epoch, pop_size=pop_size, variant=variant)
    t0 = time.perf_counter()
    g_best = model.solve(prob, seed=seed)
    dt = (time.perf_counter() - t0) * 1000.0
    cps = [int(model.checkpoints[epoch // 4]), int(model.checkpoints[epoch // 2]),
           int(model.checkpoints[3 * epoch // 4]), int(g_best.target.fitness)]
    sol = _json.dumps([float(v) for v in g_best.solution])
    return cost(decode(g_best.solution)), PY_EVALS[0], dt, sol, cps

def bench_mealpy_bbpso(seeds_json, pop_size, epoch, reps=3, variant='kennedy'):
    out = []
    for sd in _json.loads(seeds_json):
        runs = [run_mealpy_bbpso(sd, pop_size, epoch, variant) for _ in range(reps)]
        cs = [r[0] for r in runs]
        es = [r[1] for r in runs]
        ts = sorted(r[2] for r in runs)
        med = ts[len(ts) // 2] if len(ts) % 2 == 1 else (ts[len(ts) // 2 - 1] + ts[len(ts) // 2]) / 2.0
        out.append({'seed': sd, 'conflicts': cs[0], 'all_same': len(set(cs)) == 1,
                    'evals': es[0], 'ms': med, 'cp': runs[0][4], 'sol': runs[0][3]})
    return _json.dumps(out)

def time_python_evals(vecs_json, reps=5):
    import time
    vecs = _json.loads(vecs_json)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        for v in vecs:
            cost_of_vector(v)
        ts.append((time.perf_counter() - t0) * 1000.0)
    ts.sort()
    return ts[len(ts) // 2]

__mealpy_ver__ = 'mealpy ' + mealpy.__version__ + ' sur Python ' + sys.version.split()[0]");
    Console.WriteLine($"Pont PythonNet actif : {S28.Get<string>("__mealpy_ver__")}");
}

// --- Sanity check : la fonction de coût est-elle la MÊME des deux côtés ? ---
// 3 vecteurs témoins DÉTERMINISTES (LCG écrit à la main), décodés et costés des deux côtés.
public static double[] LcgVector28(int seed, int n)
{
    uint state = (uint)seed;
    var v = new double[n];
    for (int i = 0; i < n; i++)
    {
        state = state * 1664525u + 1013904223u;
        v[i] = 1.0 + (state / 4294967296.0) * 9.0; // uniforme dans [1, 10)
    }
    return v;
}

var witnessVectors = new[] { LcgVector28(1, 51), LcgVector28(2, 51), LcgVector28(3, 51) };
using (Py.GIL())
{
    S28.Set("__witness_json__",
        System.Text.Json.JsonSerializer.Serialize(witnessVectors.Select(v => v.ToList()).ToList()));
    S28.Exec(@"__py_costs__ = _json.dumps([cost_of_vector(v) for v in _json.loads(__witness_json__)])");
    var pyCosts = System.Text.Json.JsonSerializer.Deserialize<List<int>>(S28.Get<string>("__py_costs__"));
    var csCosts = witnessVectors.Select(v => CountConflicts28(DecodeR1_28(v))).ToList();
    bool identical = pyCosts.SequenceEqual(csCosts);
    Console.WriteLine($"Sanity check cout : C# {string.Join(",", csCosts)} | Python {string.Join(",", pyCosts)} " +
                      $"-> {(identical ? "IDENTIQUE" : "DIFFERENT")}");
}


Installing Packages pythonnet

Pont PythonNet actif : mealpy 3.0.2 sur Python 3.13.3


Sanity check cout : C# 67,71,60 | Python 67,71,60 -> IDENTIQUE


**Lecture.** Le pont est actif et la sanity check porte tout le bench : la fonction de coût
Python redonne *exactement* le comptage C# sur les vecteurs témoins déterministes. Le point
structurant est le **subclass `OriginalBBPSO`** : mealpy n'offre aucun PSO bare-bones (les
sept variantes du module portent toutes *w*/*c1*/*c2*), et le point d'extension officiel de la
bibliothèque est précisément la classe `Optimizer` — le jumeau hérite donc du harnais mealpy
complet (générateur seedé par `solve(prob, seed=...)`, comptage d'évaluations par `Problem`,
correction de bornes `correct_solution`, suivi `g_best`) en pinnant la formule de Kennedy.
Les deux variantes du jumeau (`kennedy`, `mgs`) ne diffèrent **que** par l'ancre et la règle
de remplacement — tout le reste (harnais, budget, graine) est identique par construction.


In [4]:
// === CELLULE FORMULES : les deux jumeaux AVANT toute mesure (exigence du cadrage paire 7) ===
// Objectif : détecter un PSO classique déguisé AVANT de mesurer, puis trancher les divergences
// d'ancre et de sélection pour que l'écart mesuré s'interprète (noyau vs portage), pas le subir.
using (Py.GIL())
{
    // 1) La détection Clerc : aucun PSO mealpy 3.0.2 n'est bare-bones — tous portent w/c1/c2.
    S28.Exec(@"__pso_catalog__ = ''
for _cls in (OriginalPSO, P_PSO, C_PSO, CL_PSO, AIW_PSO, HPSO_TVAC, LDW_PSO):
    _m = _cls(epoch=10, pop_size=5)
    _params = {k: v for k, v in _m.__dict__.items() if k in ('w', 'c1', 'c2', 'w_min', 'w_max', 'ci', 'cf', 'c_local')}
    __pso_catalog__ += f'{_cls.__name__:10s} ' + str(sorted(_params.items())) + chr(10)");
    Console.WriteLine("Catalogue PSO mealpy 3.0.2 (aucun bare-bones natif — d'ou le subclass) :");
    Console.WriteLine(S28.Get<string>("__pso_catalog__"));

    // 2) La propriété de gel du bare-bones : std = 0 => tirage déterministe = la moyenne.
    S28.Exec(@"_rng = np.random.default_rng(0)
_loc = np.array([3.5, 7.25]); _std0 = np.zeros(2)
_draws = [_rng.normal(_loc, _std0) for _ in range(4)]
__freeze__ = 'normal(loc, std=0) -> ' + str([list(map(float, d)) for d in _draws]) + ' (deterministe, egale la moyenne)'");
    Console.WriteLine($"Propriete de gel (numpy) : {S28.Get<string>("__freeze__")}");
}
Console.WriteLine(@"
Formules pinnées des jumeaux :
  MGS BareBonesParticleSwarm (in-source, DefaultSampleOperator, Box-Muller) :
      x_new ~ N((x_i + gbest)/2, |gbest - x_i|)   [ancre = position COURANTE x_i]
      selection : FitnessBasedElitistReinsertion — best-N parents+enfants (glouton pool)
  mealpy OriginalBBPSO(variant='kennedy') — Kennedy 2003 litteral :
      x_new ~ N((pbest_i + gbest)/2, |gbest - pbest_i|)   [ancre = pbest_i]
      selection : remplacement INCONDITIONNEL (la memoire vit dans pbest/gbest)
  mealpy OriginalBBPSO(variant='mgs') — aligne sur la semantique du compose MGS :
      x_new ~ N((x_i + gbest)/2, |gbest - x_i|)   [ancre = position courante]
      selection : glouton per-particle (garde le parent s'il reste meilleur)

TRANCHE : deux divergences documentees — (1) l'ANCRE (courante vs pbest), (2) la SELECTION
(best-N pool vs per-particle vs inconditionnel). L'ecart MGS<->mealpy brut les melange :
le bench croise donc TROIS bras ->
  bras 1 MGS           vs bras 2 mealpy-mgs      : memes ancre+glouton => ECART NOYAU
  bras 2 mealpy-mgs    vs bras 3 mealpy-kennedy  : meme harnais/kernel => ECART D'ANCRAGE
  bras 1 MGS           vs bras 3 mealpy-kennedy  : l'ecart brut des paires precedentes.");


Catalogue PSO mealpy 3.0.2 (aucun bare-bones natif — d'ou le subclass) :


OriginalPSO [('c1', 2.05), ('c2', 2.05), ('w', 0.4)]
P_PSO      []
C_PSO      [('c1', 2.05), ('c2', 2.05), ('w_max', 0.9), ('w_min', 0.4)]
CL_PSO     [('c_local', 1.2), ('w_max', 0.9), ('w_min', 0.4)]
AIW_PSO    [('c1', 2.05), ('c2', 2.05)]
HPSO_TVAC  [('cf', 0.1), ('ci', 0.5)]
LDW_PSO    [('c1', 2.05), ('c2', 2.05), ('w_max', 0.9), ('w_min', 0.4)]



Propriete de gel (numpy) : normal(loc, std=0) -> [[3.5, 7.25], [3.5, 7.25], [3.5, 7.25], [3.5, 7.25]] (deterministe, egale la moyenne)



Formules pinnées des jumeaux :
  MGS BareBonesParticleSwarm (in-source, DefaultSampleOperator, Box-Muller) :
      x_new ~ N((x_i + gbest)/2, |gbest - x_i|)   [ancre = position COURANTE x_i]
      selection : FitnessBasedElitistReinsertion — best-N parents+enfants (glouton pool)
  mealpy OriginalBBPSO(variant='kennedy') — Kennedy 2003 litteral :
      x_new ~ N((pbest_i + gbest)/2, |gbest - pbest_i|)   [ancre = pbest_i]
      selection : remplacement INCONDITIONNEL (la memoire vit dans pbest/gbest)
  mealpy OriginalBBPSO(variant='mgs') — aligne sur la semantique du compose MGS :
      x_new ~ N((x_i + gbest)/2, |gbest - x_i|)   [ancre = position courante]
      selection : glouton per-particle (garde le parent s'il reste meilleur)

TRANCHE : deux divergences documentees — (1) l'ANCRE (courante vs pbest), (2) la SELECTION
(best-N pool vs per-particle vs inconditionnel). L'ecart MGS<->mealpy brut les melange :
le bench croise donc TROIS bras ->
  bras 1 MGS           vs bras 2 mealpy-mg

**Lecture.** La tranche est posée *avant* la mesure, comme l'exige le cadrage : sans elle, un
écart MGS↔mealpy sur cette paire serait **ininterprétable** — il mêlerait le moteur (C#
 GéneticSharp + composé géométrique contre NumPy + harnais mealpy) et la sémantique du
portage (ancre courante contre *pbest*, glouton pool contre per-particle contre remplacement
inconditionnel). Le catalogue de la cellule établit la **détection Clerc** : les sept PSO
mealpy portent tous des constantes de vitesse — aucun ne prétend au bare-bones, et le
subclass est la seule voie honnête vers un jumeau. La propriété de gel (écart-type nul ⇒
tirage déterministe) est le comportement littéral de Kennedy : l'élite ne bouge pas tant
qu'elle est le gbest, et c'est une **propriété**, pas un bug à corriger — l'Exercice 2 en
mesurera le prix.


In [5]:
// === Moteur mealpy : course témoin (2 variantes, graine 7) + contre-vérification croisée ===
using (Py.GIL())
{
    // Échauffement symétrique (course jetée), puis courses témoins graine 7.
    S28.Exec(@"_wu_c, _wu_e, _wu_t, _wu_sol, _wu_cp = run_mealpy_bbpso(123, 50, 10, variant='mgs')
__k_c__, __k_e__, __k_t__, __k_sol__, __k_cp__ = run_mealpy_bbpso(7, 50, 160, variant='kennedy')
__m_c__, __m_e__, __m_t__, __m_sol__, __m_cp__ = run_mealpy_bbpso(7, 50, 160, variant='mgs')");
    Console.WriteLine($"mealpy BBPSO-kennedy (graine 7, témoin) : {S28.Get<int>("__k_c__")} conflits, " +
                      $"{S28.Get<int>("__k_e__")} évaluations, {S28.Get<double>("__k_t__"):F0} ms.");
    Console.WriteLine($"mealpy BBPSO-mgs (graine 7, témoin) : {S28.Get<int>("__m_c__")} conflits, " +
                      $"{S28.Get<int>("__m_e__")} évaluations, {S28.Get<double>("__m_t__"):F0} ms.");

    // Contre-vérification croisée : le vainqueur mealpy, décodé et costé côté C#.
    var solJson = S28.Get<string>("__k_sol__");
    var genes = System.Text.Json.JsonSerializer.Deserialize<double[]>(solJson);
    int csRecheck = CountConflicts28(DecodeR1_28(genes));
    Console.WriteLine($"Contre-vérif croisée : coût C# du meilleur mealpy-kennedy = {csRecheck} " +
                      $"(Python rapporte {S28.Get<int>("__k_c__")}) -> " +
                      $"{(csRecheck == S28.Get<int>("__k_c__") ? "IDENTIQUE" : "DIFFERENT")}");
}


mealpy BBPSO-kennedy (graine 7, témoin) : 26 conflits, 8050 évaluations, 600 ms.


mealpy BBPSO-mgs (graine 7, témoin) : 26 conflits, 8050 évaluations, 539 ms.


Contre-vérif croisée : coût C# du meilleur mealpy-kennedy = 26 (Python rapporte 26) -> IDENTIQUE


***

In [6]:
// === LE BENCH : 3 bras x 4 graines {0,1,7,42}, population 50, 160 générations/epochs ===
// Bras 1 MGS BBPSO | Bras 2 mealpy-mgs (sémantique alignée) | Bras 3 mealpy-kennedy (littéral 2003).
public class BenchRow28
{
    public int seed { get; set; }
    public int conflicts { get; set; }
    public bool all_same { get; set; }
    public int evals { get; set; }
    public double ms { get; set; }
    public List<int> cp { get; set; }
    public string sol { get; set; }
}

int[] Seeds28 = { 0, 1, 7, 42 };

// --- Bras 1 : côté MGS (C#), 3 répétitions par graine, ms = médiane (amendement §2) ---
var mgsRows = new List<(int seed, int conflicts, int evals, double ms, bool allSame, int[] cp)>();
foreach (var sd in Seeds28)
{
    var runs3 = new List<(int c, int e, double t, int[] q)>();
    for (int rep = 0; rep < 3; rep++)
    {
        var r = Mgs28Host.RunBbpso(sd, 50, 160);
        runs3.Add((r.Item1, r.Item2, r.Item3, r.Item5));
    }
    var times = runs3.Select(x => x.t).OrderBy(t => t).ToList();
    double med = times[1];
    mgsRows.Add((sd, runs3[0].c, runs3[0].e, med, runs3.All(x => x.c == runs3[0].c), runs3[0].q));
}

// --- Bras 2 et 3 : côté mealpy (Python, boucle unique dans le scope) ---
string mealpyMgsJson, mealpyKenJson;
using (Py.GIL())
{
    S28.Set("__seeds_json__", System.Text.Json.JsonSerializer.Serialize(Seeds28.ToList()));
    S28.Exec(@"__bench_mgs_json__ = bench_mealpy_bbpso(__seeds_json__, 50, 160, variant='mgs')
__bench_ken_json__ = bench_mealpy_bbpso(__seeds_json__, 50, 160, variant='kennedy')");
    mealpyMgsJson = S28.Get<string>("__bench_mgs_json__");
    mealpyKenJson = S28.Get<string>("__bench_ken_json__");
}
var mealpyMgsRows = System.Text.Json.JsonSerializer.Deserialize<List<BenchRow28>>(mealpyMgsJson);
var mealpyKenRows = System.Text.Json.JsonSerializer.Deserialize<List<BenchRow28>>(mealpyKenJson);

// --- Table : conflits, budget mesuré, coût par éval, checkpoints 25/50/75/100 % ---
static double Median28(List<int> xs)
{
    var s = xs.OrderBy(x => x).ToList();
    return (s.Count % 2 == 1) ? s[s.Count / 2] : (s[s.Count / 2 - 1] + s[s.Count / 2]) / 2.0;
}

Console.WriteLine($"{"moteur",-16} {"graine",6} {"conflits",9} {"evals",7} {"ms",7} {"ms/eval",8} {"cp25/50/75/100",-18}");
foreach (var r in mgsRows)
    Console.WriteLine($"{"MGS",-16} {r.seed,6} {r.conflicts,9} {r.evals,7} {r.ms,7:F0} {r.ms / r.evals,8:F3} {string.Join("/", r.cp),-18}");
foreach (var r in mealpyMgsRows)
    Console.WriteLine($"{"mealpy-mgs",-16} {r.seed,6} {r.conflicts,9} {r.evals,7} {r.ms,7:F0} {r.ms / r.evals,8:F3} {string.Join("/", r.cp),-18}");
foreach (var r in mealpyKenRows)
    Console.WriteLine($"{"mealpy-kennedy",-16} {r.seed,6} {r.conflicts,9} {r.evals,7} {r.ms,7:F0} {r.ms / r.evals,8:F3} {string.Join("/", r.cp),-18}");

var arm1 = mgsRows.Select(r => r.conflicts).ToList();
var arm2 = mealpyMgsRows.Select(r => r.conflicts).ToList();
var arm3 = mealpyKenRows.Select(r => r.conflicts).ToList();
double msEval1 = mgsRows.Average(r => r.ms / r.evals);
double msEval2 = mealpyMgsRows.Average(r => r.ms / r.evals);
double msEval3 = mealpyKenRows.Average(r => r.ms / r.evals);
Console.WriteLine();
Console.WriteLine($"MGS            : médiane conflits {Median28(arm1):F1} (min {arm1.Min()}, max {arm1.Max()}), ms/éval moyen {msEval1:F3}");
Console.WriteLine($"mealpy-mgs     : médiane conflits {Median28(arm2):F1} (min {arm2.Min()}, max {arm2.Max()}), ms/éval moyen {msEval2:F3}");
Console.WriteLine($"mealpy-kennedy : médiane conflits {Median28(arm3):F1} (min {arm3.Min()}, max {arm3.Max()}), ms/éval moyen {msEval3:F3}");
Console.WriteLine();
Console.WriteLine($"ECART NOYAU (bras1 - bras2, sémantique alignée) : médianes {Median28(arm1):F1} vs {Median28(arm2):F1}, ms/éval {msEval1 / msEval2:F2}x");
Console.WriteLine($"ECART D'ANCRAGE (bras2 - bras3, même harnais)    : médianes {Median28(arm2):F1} vs {Median28(arm3):F1}");
Console.WriteLine($"ECART BRUT (bras1 - bras3, lecture des paires)   : médianes {Median28(arm1):F1} vs {Median28(arm3):F1}, ms/éval {msEval1 / msEval3:F2}x");

// --- Classement dynamique : rangs dérivés des mesures (OrderBy), zéro récit codé en dur ---
var classement = new[] {
    ("MGS", Median28(arm1)), ("mealpy-mgs", Median28(arm2)), ("mealpy-kennedy", Median28(arm3))
}.OrderBy(x => x.Item2).ToList();
Console.WriteLine();
Console.WriteLine("Classement qualité (médiane conflits croissante) : " +
                  string.Join(" < ", classement.Select(x => $"{x.Item1} ({x.Item2:F1})")));

// --- Forme de l'écart aux checkpoints : MGS - mealpy-mgs (noyau) et mgs - kennedy (ancrage) ---
// delta < 0 => le premier bras est DEVANT (moins de conflits) à ce checkpoint.
static string ShapeOf(int[] d)
{
    if (d.All(x => x == 0)) return "nul (identiques)";
    var sgn = d.Select(x => x == 0 ? 0 : (x < 0 ? -1 : 1)).ToList();
    bool earlySame = sgn.Take(2).All(x => x == sgn[0]) && sgn[0] != 0;
    if (earlySame && sgn.All(x => x == sgn[0])) return sgn[0] < 0 ? "persistant (bras A devant)" : "persistant (bras B devant)";
    if (earlySame && sgn.Last() != 0 && sgn.Last() != sgn[0]) return "croisement";
    if (sgn.Last() == 0) return "refermeture";
    return "mixte";
}

Console.WriteLine();
Console.WriteLine("Forme noyau (MGS - mealpy-mgs) et ancrage (mealpy-mgs - mealpy-kennedy), par graine :");
for (int i = 0; i < Seeds28.Length; i++)
{
    var dn = Enumerable.Range(0, 4).Select(k => mgsRows[i].cp[k] - mealpyMgsRows[i].cp[k]).ToArray();
    var da = Enumerable.Range(0, 4).Select(k => mealpyMgsRows[i].cp[k] - mealpyKenRows[i].cp[k]).ToArray();
    Console.WriteLine($"  graine {Seeds28[i],2} : noyau [{string.Join(",", dn)}] {ShapeOf(dn),-28} | " +
                      $"ancrage [{string.Join(",", da)}] {ShapeOf(da)}");
}

int detMgs = mgsRows.Count(r => r.allSame) + mealpyMgsRows.Count(r => r.all_same) + mealpyKenRows.Count(r => r.all_same);
Console.WriteLine();
Console.WriteLine($"Déterminisme (3 répétitions identiques par graine) : {detMgs}/12 bras-graines stables.");


moteur           graine  conflits   evals      ms  ms/eval cp25/50/75/100    


MGS                   0        31    8000     292    0,036 33/31/31/31       


MGS                   1        31    8000     289    0,036 31/31/31/31       


MGS                   7        29    8000     302    0,038 29/29/29/29       


MGS                  42        30    8000     283    0,035 30/30/30/30       


...

  graine  0 : noyau [1,9,11,12] persistant (bras B devant)   | ancrage [0,0,0,0] nul (identiques)


  graine  1 : noyau [0,5,6,6] mixte                        | ancrage [0,0,0,0] nul (identiques)


  graine  7 : noyau [-3,3,3,3] mixte                        | ancrage [0,0,0,0] nul (identiques)


  graine 42 : noyau [-2,0,2,3] mixte                        | ancrage [0,0,0,0] nul (identiques)


Déterminisme (3 répétitions identiques par graine) : 12/12 bras-graines stables.


In [7]:
// === Coût par évaluation : la fitness seule, hors moteur, 500 vecteurs identiques ===
// Les vecteurs sont générés côté C# (LCG, graines 42..541) et passés en JSON au Python :
// les DEUX côtés chronomètrent decode+coût sur exactement les mêmes 500 points.
int K28 = 500;
var benchVecs = new List<double[]>();
for (int i = 0; i < K28; i++) benchVecs.Add(LcgVector28(42 + i, 51));

var csTimes = new List<double>();
for (int rep = 0; rep < 5; rep++)
{
    var swRep = Stopwatch.StartNew();
    foreach (var v in benchVecs) CountConflicts28(DecodeR1_28(v));
    swRep.Stop();
    csTimes.Add(swRep.Elapsed.TotalMilliseconds);
}
csTimes.Sort();
double csMs = csTimes[2]; // médiane de 5 (amendement §2)

double pyMs;
using (Py.GIL())
{
    S28.Set("__vecs_json__", System.Text.Json.JsonSerializer.Serialize(benchVecs.Select(v => v.ToList()).ToList()));
    S28.Exec(@"__py_ms__ = time_python_evals(__vecs_json__)");
    pyMs = S28.Get<double>("__py_ms__");
}

Console.WriteLine($"Fitness seule, {K28} vecteurs identiques (médiane de 5 répétitions par côté) :");
Console.WriteLine($"  C#     : {csMs:F1} ms total -> {csMs / K28:F3} ms/éval");
Console.WriteLine($"  Python : {pyMs:F1} ms total -> {pyMs / K28:F3} ms/éval");
Console.WriteLine($"  rapport Python/C# : {pyMs / csMs:F2}x");


Fitness seule, 500 vecteurs identiques (médiane de 5 répétitions par côté) :


  C#     : 4,1 ms total -> 0,008 ms/éval


  Python : 18,8 ms total -> 0,038 ms/éval


  rapport Python/C# : 4,59x


**Lecture du croisement.**

- **Classement** : les deux jumeaux mealpy ex æquo en tête (médiane 25,5 conflits chacun), MGS troisième (30,5). L'écart brut de 5 conflits est donc imputable au moteur, pas au portage.
- **Écart d'ancrage nul — la paire la plus propre de l'Epic** : les deux jumeaux mealpy (ancre `local_solution` = pbest contre ancre `solution` = position courante, remplacement inconditionnel contre glouton) produisent des trajectoires **identiques graine par graine** — mêmes conflits finaux et mêmes checkpoints sur les 4 graines (écarts `[0,0,0,0]` partout). La divergence tranchée dans la cellule formules ne change rien ici : sur ce problème, la mise à jour locale gloutonne confond pbest et position courante dès qu'une particule s'améliore.
- **Écart noyau réel, 5 conflits pour mealpy** : à sémantique strictement alignée (le bras mealpy-mgs porte exactement la formule MGS dans le harnais mealpy), mealpy atteint 25,5 contre 30,5 — la différence vient du moteur (harnais mealpy : évaluation initiale de la population, `update_global_best` post-epoch), pas des équations. La forme le confirme : MGS est devant ou égal au premier quart (checkpoints 29-33 contre 31-32), puis mealpy creuse l'écart de mi-course à la fin — graine 0 persistante `[1,9,11,12]`, graines 1/7/42 en croisement. Concrètement, mealpy continue d'exploiter en fin de course (20→19 et 28→27 entre 75 % et 100 %) quand MGS est figé dès le premier quart (cp25 = cp100 sur 3 graines sur 4) — la propriété de **gel** de la cellule formules, visible dans la sortie.
- **Vitesse** : MGS reste 2× moins cher par évaluation (0,036 contre 0,073 ms/éval, ratio 0,50×) et la fitness C# isolée est 4,59× plus rapide — mais le budget est apparié (8 000/8 050 évaluations), donc mealpy gagne à budget égal. C'est le profil inversé du FBI (MGS-27) : là-bas le portage dérivait et gagnait des deux mains ; ici le portage est fidèle, ne dérive pas — et c'est le harnais adverse qui paie la qualité.
- **Fil rouge PSO de l'Epic** : MGS-22 (PSO classique à coefficients) donnait mealpy devant en qualité à ex æquo en vitesse ; MGS-28 (BBPSO, zéro coefficient) resserre la démonstration — dépouiller la particule de w/c1/c2 ne change pas le verdict moteur, il l'isole : tout ce qui reste de l'écart est noyau.

## Exercice 1 : budget ×4 — l'écart se referme-t-il ou s'installe-t-il ?

Le croisement s'est joué à budget 8 000 évaluations. Refaites-le à budget ×4 (population 50,
640 générations/epochs, 4 graines, les trois bras) et comparez : l'écart de médiane se
referme, se stabilise, ou s'agrandit ? Relevez aussi les checkpoints 25/50/75/100 % — la
*forme* de l'écart à budget long est-elle la même qu'à court ?

**Indice** : côté MGS, `Mgs28Host.RunBbpso(sd, 50, 640)` renvoie déjà les checkpoints ; côté
mealpy, `bench_mealpy_bbpso(seeds, 50, 640, variant=...)` fait de même.


In [8]:
// EXERCICE 1 : budget x4 (pop 50, 640 générations/epochs), 4 graines, les trois bras.
// Décommentez et exécutez :
// foreach (var sd in new[] {0, 1, 7, 42})
// {
//     var r = Mgs28Host.RunBbpso(sd, 50, 640);
//     Console.WriteLine($"MGS BBPSO x4 (graine {sd}) : {r.Item1} conflits, {r.Item2} évaluations, " +
//                       $"{r.Item3:F0} ms, cp = {string.Join("/", r.Item5)}.");
// }
// using (Py.GIL())
// {
//     S28.Set("__seeds_json__", System.Text.Json.JsonSerializer.Serialize(new[] {0, 1, 7, 42}.ToList()));
//     S28.Exec(@"__bx4_mgs__ = bench_mealpy_bbpso(__seeds_json__, 50, 640, variant='mgs')
// __bx4_ken__ = bench_mealpy_bbpso(__seeds_json__, 50, 640, variant='kennedy')");
//     Console.WriteLine(S28.Get<string>("__bx4_mgs__"));
//     Console.WriteLine(S28.Get<string>("__bx4_ken__"));
// }

Console.WriteLine("Exercice a completer (decommentez le bloc ci-dessus).");


Exercice a completer (decommentez le bloc ci-dessus).


## Exercice 2 : lever le gel — un plancher d'écart-type change-t-il la donne ?

Le bare-bones **gèle** au meilleur global : quand `ancre == gbest`, l'écart-type s'annule et
la particule-élite ne peut plus bouger. Ajoutez au subclass un plancher d'écart-type
`std_floor` (`std = np.maximum(np.abs(g - anchor), std_floor)`) et mesurez les trois bras à
`std_floor` ∈ {0 (statu quo), 0,25, 0,5} : le gel est-il une propriété qui *porte* la
convergence, ou un frein que l'on lève gratuitement ?

**Indice** : la modification tient en une ligne dans `evolve` — pensez à documenter le
changement de sémantique dans la docstring du subclass, comme le fait le portage MGS pour
son ancre courante.


In [9]:
// EXERCICE 2 : plancher d'ecart-type dans le subclass mealpy (std_floor).
// Décommentez et exécutez (esquisse — le subclass OriginalBBPSO est déjà dans le scope S28) :
// using (Py.GIL())
// {
//     S28.Exec(@"class BBPSOFloor(OriginalBBPSO):
//     def __init__(self, epoch=10000, pop_size=100, variant='kennedy', std_floor=0.25, **kwargs):
//         super().__init__(epoch, pop_size, variant, **kwargs)
//         self.std_floor = std_floor
//     def evolve(self, epoch):
//         if epoch - 1 in (self.epoch // 4, self.epoch // 2, 3 * self.epoch // 4):
//             self.checkpoints[epoch - 1] = float(self.g_best.target.fitness)
//         for idx in range(self.pop_size):
//             anchor = self.pop[idx].local_solution if self.variant == 'kennedy' else self.pop[idx].solution
//             g = self.g_best.solution
//             mean = 0.5 * (anchor + g)
//             std = np.maximum(np.abs(g - anchor), self.std_floor)
//             pos_new = self.correct_solution(self.generator.normal(mean, std))
//             target = self.get_target(pos_new)
//             if self.variant == 'kennedy' or self.compare_target(target, self.pop[idx].target, self.problem.minmax):
//                 self.pop[idx].update(solution=pos_new.copy(), target=target.copy())
//             if self.compare_target(target, self.pop[idx].local_target, self.problem.minmax):
//                 self.pop[idx].update(local_solution=pos_new.copy(), local_target=target.copy())
// def run_floor(seed, floor):
//     PY_EVALS[0] = 0
//     prob = SudokuProblem(bounds=FloatVar(lb=(1.0,) * len(empties), ub=(10.0,) * len(empties), name='genes'), minmax='min')
//     model = BBPSOFloor(epoch=160, pop_size=50, variant='kennedy', std_floor=floor)
//     g_best = model.solve(prob, seed=seed)
//     return cost(decode(g_best.solution))
// __floor_demo__ = str({f: [run_floor(sd, f) for sd in (0, 1, 7, 42)] for f in (0.0, 0.25, 0.5)})");
//     Console.WriteLine(S28.Get<string>("__floor_demo__"));
// }

Console.WriteLine("Exercice a completer (decommentez le bloc ci-dessus).");


Exercice a completer (decommentez le bloc ci-dessus).


## Exercice 3 : profiler la fitness — où va la milliseconde ?

Le rapport ms/éval mêle le tirage gaussien, la correction de bornes et le coût Sudoku. Comme
dans les paires précédentes, séparez le **décodage** du **comptage de conflits** sur les 500
vecteurs témoins, des deux côtés du pont : la fitness C# reste-t-elle plusieurs fois plus
rapide, et la part decode/coût change-t-elle de côté ?


In [10]:
// EXERCICE 3 : profil decode vs cost, 500 vecteurs, deux côtés.
// Décommentez et exécutez (adapté de MGS-22/23 exercice 3) :
// var decSw = Stopwatch.StartNew();
// foreach (var v in benchVecs) DecodeR1_28(v);
// decSw.Stop();
// Console.WriteLine($"C# decode seul : {decSw.Elapsed.TotalMilliseconds / K28:F3} ms/vec " +
//     $"(reste = coût : {(csMs - decSw.Elapsed.TotalMilliseconds) / K28:F3} ms/vec)");
// using (Py.GIL())
// {
//     S28.Set("__vecs_json__", System.Text.Json.JsonSerializer.Serialize(benchVecs.Select(v => v.ToList()).ToList()));
//     S28.Exec(@"import time
// _vecs = _json.loads(__vecs_json__)
// _t0 = time.perf_counter()
// _grids = [decode(v) for v in _vecs]
// _t1 = time.perf_counter()
// _csts = [cost(g) for g in _grids]
// _t2 = time.perf_counter()
// __py_profil__ = f'Python decode seul {(_t1 - _t0) * 1000.0 / len(_vecs):.3f} ms/vec, coût {(_t2 - _t1) * 1000.0 / len(_vecs):.3f} ms/vec'");
//     Console.WriteLine(S28.Get<string>("__py_profil__"));
// }

Console.WriteLine("Exercice a completer (decommentez le bloc ci-dessus).");


Exercice a completer (decommentez le bloc ci-dessus).
